In [2]:
# Import thư viện
import pandas as pd
import os
# Bước 1: Đọc file all_reviews.csv
print("Bước 1: Đọc dữ liệu từ all_reviews.csv")
df = pd.read_csv('data/all_reviews.csv')
print("Số hàng và cột trong df:", df.shape)
print("Thông tin cơ bản của df:")
print(df.info())
print("\nThống kê mô tả của df:")
print(df.describe(include='all'))
print("\nSố người dùng duy nhất:", df['username'].nunique())
print("\nSố sản phẩm duy nhất:", df['id_product'].nunique())
print("-" * 50)

Bước 1: Đọc dữ liệu từ all_reviews.csv
Số hàng và cột trong df: (40232, 4)
Thông tin cơ bản của df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40232 entries, 0 to 40231
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_product    40232 non-null  object
 1   name_product  40232 non-null  object
 2   username      40178 non-null  object
 3   rating        40232 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.2+ MB
None

Thống kê mô tả của df:
                                              id_product     name_product  \
count                                              40232            40232   
unique                                              2502             2490   
top     pin-sac-du-phong-polymer-10000mah-12w-ava-ds609a  AVA+ 12W DS609A   
freq                                                 681              681   
mean                                                 NaN              NaN   

In [3]:
# Bước 2: Đếm số rating mỗi người dùng
print("Bước 2: Đếm số rating mỗi người dùng")
ratings_per_user = df.groupby('username').size().reset_index(name='ratings_count')
print("Số người dùng:", len(ratings_per_user))
print("Thống kê số rating mỗi người dùng:")
print(ratings_per_user['ratings_count'].describe())
print("\nPhân bố số rating mỗi người dùng (top 10):")
print(ratings_per_user['ratings_count'].value_counts().sort_index().head(10))
print("-" * 50)

Bước 2: Đếm số rating mỗi người dùng
Số người dùng: 21147
Thống kê số rating mỗi người dùng:
count    21147.000000
mean         1.899939
std          4.989558
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        150.000000
Name: ratings_count, dtype: float64

Phân bố số rating mỗi người dùng (top 10):
ratings_count
1     16090
2      2874
3       892
4       398
5       194
6       126
7        96
8        69
9        37
10       36
Name: count, dtype: int64
--------------------------------------------------


In [4]:
# Bước 3: Lọc người dùng có từ 10 rating trở lên
print("Bước 3: Lọc người dùng có từ 10 rating trở lên")
min_ratings = 10
filtered_users = ratings_per_user[ratings_per_user['ratings_count'] >= min_ratings]
print("Số người dùng có từ 10 rating trở lên:", len(filtered_users))
print("Thống kê số rating của người dùng sau khi lọc:")
print(filtered_users['ratings_count'].describe())
print("\nDanh sách người dùng và số rating (top 5):")
print(filtered_users.head())
print("-" * 50)

Bước 3: Lọc người dùng có từ 10 rating trở lên
Số người dùng có từ 10 rating trở lên: 371
Thống kê số rating của người dùng sau khi lọc:
count    371.000000
mean      29.080863
std       24.689508
min       10.000000
25%       12.000000
50%       19.000000
75%       37.000000
max      150.000000
Name: ratings_count, dtype: float64

Danh sách người dùng và số rating (top 5):
    username  ratings_count
158   A Dũng             11
201   A Long             10
260  A Thành             11
362       An             54
378      Anh             49
--------------------------------------------------


In [5]:
# Bước 4: Giữ lại các đánh giá của người dùng có từ 10 rating
print("Bước 4: Giữ lại các đánh giá của người dùng có từ 10 rating")
filtered_ratings_df = df[df['username'].isin(filtered_users['username'])]
print("Số hàng và cột trong filtered_ratings_df:", filtered_ratings_df.shape)
print("Số người dùng duy nhất sau lọc:", filtered_ratings_df['username'].nunique())
print("Số sản phẩm duy nhất sau lọc:", filtered_ratings_df['id_product'].nunique())
print("-" * 50)

Bước 4: Giữ lại các đánh giá của người dùng có từ 10 rating
Số hàng và cột trong filtered_ratings_df: (10789, 4)
Số người dùng duy nhất sau lọc: 371
Số sản phẩm duy nhất sau lọc: 1875
--------------------------------------------------


In [6]:
# Bước 5: Chia tập train/test (5 đánh giá mỗi user cho test)
print("Bước 5: Chia tập train/test")
test_df = pd.DataFrame()
train_df = pd.DataFrame()
for username in filtered_users['username']:
    user_ratings = filtered_ratings_df[filtered_ratings_df['username'] == username]
    if len(user_ratings) >= 5:  # Đảm bảo đủ 5 rating để chia
        test_sample = user_ratings.sample(n=5, random_state=42)
        train_sample = user_ratings.drop(test_sample.index)
        test_df = pd.concat([test_df, test_sample])
        train_df = pd.concat([train_df, train_sample])
print("Số hàng trong tập test:", len(test_df))
print("Số hàng trong tập train:", len(train_df))
print("Số người dùng trong tập test:", test_df['username'].nunique())
print("Số người dùng trong tập train:", train_df['username'].nunique())
print("-" * 50)

Bước 5: Chia tập train/test
Số hàng trong tập test: 1855
Số hàng trong tập train: 8934
Số người dùng trong tập test: 371
Số người dùng trong tập train: 371
--------------------------------------------------


In [10]:
# Bước 6: Gộp mô tả khách sạn trong tập train cho mỗi user
print("Bước 6: Gộp mô tả khách sạn trong tập train")
products_df = pd.read_csv('data/product/products_cleaned.csv')
user_descriptions = train_df.merge(products_df[['id_product', 'description']], on='id_product')
user_descriptions = user_descriptions.groupby('username')['description'].apply(' '.join).reset_index()
print("Số người dùng sau gộp mô tả:", len(user_descriptions))
print("Mẫu 5 dòng user_descriptions:")
print(user_descriptions.head())
print("-" * 50)

Bước 6: Gộp mô tả khách sạn trong tập train
Số người dùng sau gộp mô tả: 371
Mẫu 5 dòng user_descriptions:
  username                                        description
0   A Dũng  Thiết kế tinh tế, hiện đại, dành riêng cho các...
1   A Long  Loa thanh Sony HT-S100F 120W cung cấp âm thanh...
2  A Thành  Thiết kế hiện đại, trẻ trung, dành riêng cho c...
3       An  Camera IP 360 Độ 2MP Ezviz C6N nổi bật với khả...
4      Anh  Camera giám sát bao quát tốt không gian với gó...
--------------------------------------------------


In [11]:
# Bước 7: Ghi tập test vào file
print("Bước 7: Ghi tập test vào test_data.csv")
test_output_path = '../data/test_data.csv'
os.makedirs(os.path.dirname(test_output_path), exist_ok=True)
test_df.to_csv(test_output_path, index=False)
print(f"Tập test đã được ghi vào {test_output_path}")
print("Kiểm tra 5 dòng đầu của tập test:")
print(test_df.head())
print("Số hàng trong tập test:", len(test_df))
print("-" * 50)

Bước 7: Ghi tập test vào test_data.csv
Tập test đã được ghi vào ../data/test_data.csv
Kiểm tra 5 dòng đầu của tập test:
                                     id_product  \
21193       loa-keo-karaoke-mobell-mk-6080-500w   
3522                 chuot-khong-day-zadez-m338   
37629  tai-nghe-bluetooth-chup-tai-havit-h663bt   
38712  tai-nghe-bluetooth-chup-tai-havit-h663bt   
13286          mvw-ms070-02-nam?utm_flashsale=1   

                         name_product username  rating  
21193          Loa kéo Mobell MK-6080   A Dũng       1  
3522       Chuột Không dây Zadez M338   A Dũng       5  
37629  Tai nghe Chụp Tai Havit H663BT   A Dũng       1  
38712  Tai nghe Chụp Tai Havit H663BT   A Dũng       1  
13286          MVW 40 mm Nam MS070-02   A Dũng       4  
Số hàng trong tập test: 1855
--------------------------------------------------


In [12]:
# Bước 8: Ghi tập train vào file
print("Bước 8: Ghi tập train vào train_data.csv")
train_output_path = '../data/train_data.csv'
os.makedirs(os.path.dirname(train_output_path), exist_ok=True)
train_df.to_csv(train_output_path, index=False)
print(f"Tập train đã được ghi vào {train_output_path}")
print("Kiểm tra 5 dòng đầu của tập train:")
print(train_df.head())
print("Số hàng trong tập train:", len(train_df))

Bước 8: Ghi tập train vào train_data.csv
Tập train đã được ghi vào ../data/train_data.csv
Kiểm tra 5 dòng đầu của tập train:
                                       id_product  \
7280   casio-mtp-v004gl-7audf-nam?utm_flashsale=1   
15454                    citizen-nh8360-12a-trang   
19393                           dalton-ts-15g600x   
27468                                 mobell-f209   
32805             kidcare-s88-den?utm_flashsale=1   

                                  name_product username  rating  
7280        CASIO 41.5 mm Nam MTP-V004GL-7AUDF   A Dũng       5  
15454  CITIZEN Mechanical 41 mm Nam NH8360-12A   A Dũng       5  
19393                Loa kéo Dalton TS-15G600X   A Dũng       4  
27468                           Mobell F209 4G   A Dũng       4  
32805          Kidcare S88 43.4mm dây silicone   A Dũng       5  
Số hàng trong tập train: 8934


In [1]:
import pandas as pd

def remove_duplicate_products(input_file, output_file):
    try:
        # Đọc file CSV
        products_df = pd.read_csv(input_file)
        
        # Đảm bảo id_product là chuỗi
        products_df['id_product'] = products_df['id_product'].astype(str)
        
        # Kiểm tra trùng lặp trong id_product
        if products_df['id_product'].duplicated().any():
            duplicates = products_df[products_df['id_product'].duplicated(keep=False)]['id_product'].unique()
            print(f"Tìm thấy {len(duplicates)} id_product trùng lặp: {list(duplicates)}")
            print("Giữ bản ghi đầu tiên cho mỗi id_product.")
            # Loại bỏ trùng lặp, giữ bản ghi đầu tiên
            products_df = products_df.drop_duplicates(subset='id_product', keep='first')
        else:
            print("Không tìm thấy id_product trùng lặp.")
        
        # Lưu file mới
        products_df.to_csv(output_file, index=False)
        print(f"Đã lưu file sạch tại: {output_file}")
        print(f"Số bản ghi sau khi loại bỏ trùng lặp: {len(products_df)}")
        
        # Trả về DataFrame đã làm sạch để kiểm tra
        return products_df
    
    except FileNotFoundError:
        print(f"Không tìm thấy file: {input_file}")
        return None
    except Exception as e:
        print(f"Lỗi xảy ra: {str(e)}")
        return None

if __name__ == "__main__":
    input_file = 'data/product/products.csv'  # Đường dẫn file đầu vào
    output_file = 'data/product/products_cleaned.csv'  # Đường dẫn file đầu ra
    cleaned_df = remove_duplicate_products(input_file, output_file)
    if cleaned_df is not None:
        print("Mẫu 5 bản ghi đầu tiên sau khi làm sạch:")
        print(cleaned_df.head())

Tìm thấy 131 id_product trùng lặp: ['adapter-chuyen-doi-usb-c-6-in-1-ugreen-60384', 'adapter-chuyen-doi-type-c-4-in-1-xmobile-ds606h', 'adapter-chuyen-doi-type-c-4-in-1-xmobile-ds122f', 'adapter-chuyen-doi-usb-type-c-4-in-1-hyperdrive-hd41', 'cap-chuyen-doi-type-c-sang-35mm-apple-mu7e2-trang', 'adapter-chuyen-doi-usb-c-5-in-1-baseus-ultrajoy-bs-oh150', 'adapter-type-c-sang-hdmi-type-c-usb-apple-muf82', 'adapter-chuyen-doi-type-c-4-in-1-hyperdrive-next-hd4001', 'adapter-chuyen-doi-type-c-6-in-1-hyperdrive-next-hd4002', 'adapter-chuyen-doi-type-c-7-in-1-hyperdrive-next-hd4003', 'adapter-chuyen-doi-type-c-8-in-1-hyperdrive-next-hd4004', 'adapter-chuyen-doi-usb-c-6-in-1-hyperdrive-hd22e', 'adapter-chuyen-doi-type-c-10-in-1-hyperdrive-next-hd4005', 'huawei-watch-fit-4', 'huawei-watch-fit-4-pro', 'huawei-watch-fit-4-pro-day-nylon', 'huawei-watch-gt-5-46mm-vien-thep-day-cao-su', 'huawei-watch-gt-5-pro-46mm-vien-titanium-day-cao-su', 'huawei-watch-gt-5', 'huawei-watch-d2-day-cao-su', 'huawei-w

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from underthesea import word_tokenize
import matplotlib.pyplot as plt

# Dữ liệu từ product.csv
data = """id_product,name_product,description,category,img
asus-vivobook-go-15-e1504fa-r5-nj776w,Asus Vivobook Go 15 E1504FA R5 7520U (NJ776W),"Laptop Asus Vivobook Go 15 E1504FA R5 7520U (NJ776W) mang phong cách thiết kế sang trọng, hiệu năng mạnh mẽ cùng tính đa năng sử dụng, chắc chắn sẽ giúp bạn đáp ứng mọi tác vụ công việc và học tập hàng ngày một cách hiệu quả và chuyên nghiệp nhất. Thiết kế quen thuộc, kiểu cách sang trọng Kiểu dáng đã quá quen thuộc đến từ các dòng Vivobook nhà Asus, tuy vậy nhưng với thiết kế ngoại hình tối giản hiện đại như vậy, cá nhân mình lại nhận thấy cực kì phù hợp với xu hướng thời trang hiện nay. Laptop Asus vẫn giữ được nét thuần tuý với gam màu bạc sáng khá thu hút, vỏ được chế tác bằng nhựa nhưng lại rất cứng cáp, độ bền lại còn được đảm bảo chuẩn quân đội MIL STD 810H và các bề mặt được gia công ghép nối rất kĩ, nên mình chỉ việc trang bị thêm một chiếc túi chống sốc là có thể an tâm mang theo mọi nơi rồi. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/311178/asus-vivobook-go-15-e1504fa-r5-nj776w-140225-100949-251-600x600.jpg
hp-15-fd0234tu-i5-9q969pa,HP 15 fd0234TU i5 1334U (9Q969PA),"Tích hợp tuyệt vời với đa dạng chế độ làm việc trong ngày, đẩy cao hiệu suất tư duy sáng tạo cũng như nâng hiệu quả công việc của bạn lên cao nhất với laptop HP 15 fd0234TU i5 1334U (9Q969PA), được tích hợp con chip Gen 13 hiện đại, khối lượng nhẹ nhàng cùng cấu trúc cực kì dễ dàng mang theo bên mình. • Bộ đôi hiệu năng ổn định Intel Core i5 1334U và card tích hợp Intel Iris Xe Graphics cho phép người dùng hoàn thành công việc văn phòng trên Word, chạy hàng trăm hàng tính toán trên Excel hay Sheet mà không bị giật lag gián đoạn, thậm chí người dùng cũng có thể làm việc hiệu quả đối với các phần mềm thiết kế hình ảnh, edit video cơ bản và chiến game nhẹ. • Với RAM 16 GB trên laptop bạn có thể mở nhiều tab trên trình duyệt, chạy nhiều chương trình cùng một lúc mà không bị chậm hay lag máy. Việc hỗ trợ nâng cấp lên đến cũng đảm bảo nhiều lượng tài nguyên hơn cho quá trình vận hành. Ổ cứng có khả năng đọc, ghi dữ liệu nhanh hơn so với các loại ổ cứng HDD cũ, khiến mọi thao tác đều trở nên mượt mà hơn. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/323920/hp-15-fd0234tu-i5-9q969pa-170225-105831-192-600x600.jpg
dell-inspiron-15-3520-i5-n5i5057w1,Dell Inspiron 15 3520 i5 1235U (N5I5057W1),"Laptop Dell Inspiron 15 3520 i5 1235U (N5I5057W1) là laptop tầm trung lý tưởng cho văn phòng, học sinh, sinh viên và người dùng cần thiết bị linh hoạt cho công việc và giải trí. Với vi xử lý Intel Core i5 thế hệ 12, máy mang đến hiệu suất ổn định, khả năng đa nhiệm mượt mà và thiết kế gọn nhẹ, phù hợp cho mọi nhu cầu sử dụng hàng ngày. Thiết kế tối giản, độ bền ấn tượng Laptop được ""xây dựng"" với chất liệu nhựa cao cấp, bề mặt được hoàn thiện tỉ mỉ, giúp máy bền bỉ hơn trước các tác động nhẹ hoặc di chuyển. Khối lượng chỉ 1.66 kg, khiến việc di chuyển trở nên dễ dàng mà không gây cảm giác nặng nề, phù hợp cho người dùng làm việc suốt cả ngày. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/333886/dell-inspiron-15-3520-i5-n5i5057w1-638774726911323154-600x600.jpg
lenovo-ideapad-slim-3-15irh10-i5-83k1000hvn,Lenovo IdeaPad Slim 3 15IRH10 i5 13420H (83K1000HVN),"Laptop Lenovo IdeaPad Slim 3 15IRH10 i5 13420H (83K1000HVN) là sự lựa chọn lý tưởng cho học sinh, sinh viên và nhân viên văn phòng nhờ sự kết hợp hoàn hảo giữa hiệu năng ổn định, thiết kế mỏng nhẹ và tính di động cao. Đây là người bạn đồng hành đáng tin cậy, đáp ứng tốt mọi nhu cầu từ học tập, làm việc đến giải trí, mang đến trải nghiệm sử dụng tuyệt vời trong tầm giá. Sức mạnh vượt trội, cân mọi tác vụ với chip Intel Core i5 thế hệ 13 ""Trái tim"" của chiếc laptop học tập - văn phòng này là bộ vi xử lý Intel Core i5 Raptor Lake - 13420H với 8 nhân và 12 luồng, tốc độ tối đa lên đến 4.6 GHz nhờ Turbo Boost. Sức mạnh này cho phép máy xử lý mượt mà các tác vụ văn phòng hàng ngày như soạn thảo văn bản, duyệt web, làm việc với bảng tính, cũng như các ứng dụng đồ họa nhẹ và giải trí đa phương tiện. Card đồ họa tích hợp cũng đủ sức đáp ứng nhu cầu xem phim, chơi game online nhẹ nhàng. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/334442/lenovo-ideapad-slim-3-15irh10-i5-83k1000hvn-638775478046964172-600x600.jpg
macbook-air-13-inch-m4-16gb-256gb,MacBook Air 13 inch M4 16GB/256GB,"Không làm người dùng thất vọng, Apple đã cho ra mắt MacBook Air M4 16GB không chỉ là một chiếc laptop siêu mỏng nhẹ mà còn mang đến hiệu suất mạnh mẽ với chip Apple M4 và RAM 16 GB, cùng màn hình Liquid Retina rực rỡ và thời lượng pin ấn tượng. Sản phẩm này là sự lựa chọn hoàn hảo cho mọi nhu cầu, phù hợp cho cả người dùng văn phòng, sinh viên, đặc biệt là nhà sáng tạo nội dung, thiết kế đồ hoạ. Hiệu suất đồ họa ấn tượng Bộ vi xử lý Apple M4 được sản xuất trên tiến trình tiên tiến của Apple, giúp tối ưu hiệu suất và tiết kiệm năng lượng vượt trội. Với 10 nhân CPU, con chip này mang lại khả năng xử lý nhanh chóng và mượt mà cho mọi tác vụ, từ công việc văn phòng, lập trình, chỉnh sửa ảnh đến biên tập video. Xem thêm",laptop,https://cdn.tgdd.vn/Products/Images/44/335362/macbook-air-13-inch-m4-vang-600x600.jpg
acer-aspire-go-ag15-31p-30m4-i3-nxkrpsv004,Acer Aspire Go AG15 31P 30M4 i3 N305 (NX.KRPSV.004),"Laptop Acer Aspire Go AG15 31P 30M4 i3 N305 (NX.KRPSV.004) là một chiếc laptop sở hữu thiết kế gọn nhẹ, hiệu suất ổn định cùng màn hình lớn, đáp ứng hoàn hảo nhu cầu học tập, làm việc và giải trí nhẹ nhàng. Đây là lựa chọn lý tưởng cho sinh viên, nhân viên văn phòng hoặc những ai đang tìm kiếm một thiết bị đáng tin cậy với mức giá hợp lý. Sức mạnh tiềm ẩn bên trong vẻ ngoài thanh lịch Laptop Acer được trang bị CPU Intel Core i3 Alder Lake chuỗi N - N305 với 8 nhân và 8 luồng, cho phép xử lý mượt mà các tác vụ hàng ngày như soạn thảo văn bản, làm việc trên Excel, duyệt web hay tham gia các cuộc họp trực tuyến. Nhờ xung nhịp tối đa lên đến 3.8 GHz, hiệu năng của máy đủ để đảm bảo sự ổn định trong quá trình sử dụng mà vẫn tiết kiệm điện năng. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/334997/acer-aspire-go-ag15-31p-30m4-i3-nxkrpsv004-thumb-638828183673538656-600x600.jpg
hp-15-fc0085au-r5-a6vv8pa,HP 15 fc0085AU R5 7430U (A6VV8PA),"Nổi bật và quá thân quen trong phân khúc laptop học tập - văn phòng giá rẻ, chiếc laptop HP 15 fc0085AU R5 7430U (A6VV8PA) với cấu hình ổn định, vận hành hiệu quả mọi tác vụ từ làm việc đến giải trí đa phương tiện. Máy hội tụ đầy đủ các yếu tố để trở thành bạn trợ thủ lý tưởng cho người dùng. • Laptop HP với vi xử lý AMD Ryzen 5 - 7430U kết hợp cùng card AMD Radeon Graphics giúp đảm nhận mượt mà từ công việc văn phòng đến giải trí, giải quyết nhanh tài liệu với Microsoft Office và Google Docs. Hơn nữa, một số công việc như thiết kế đồ hoạ thông dụng với Adobe Photoshop, Figma,... • Bộ nhớ RAM 16 GB DDR4 đảm bảo khả năng đa nhiệm mượt mà, phù hợp cho công việc đòi hỏi nhiều tài nguyên như đa nhiệm nhiều cửa sổ công việc, thiết kế đồ hoạ cơ bản mà không lo giật lag, gián đoạn. Ổ cứng với dung lượng cung cấp không gian lưu trữ rộng rãi cho nhiều ứng dụng và tệp tin khác nhau. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/327098/hp-15-fc0085au-r5-a6vv8pa-170225-110652-878-600x600.jpg
asus-vivobook-15-x1504va-i3-nj1634w,Asus Vivobook 15 X1504VA i3 1315U (NJ1634W),"Một chiếc laptop lý tưởng cho học tập và làm việc không chỉ cần hiệu năng mạnh mẽ mà còn phải có thiết kế tinh tế, hiện đại cùng mức giá hợp lý. Laptop Asus Vivobook 15 X1504VA i3 1315U (NJ1634W) chính là sự lựa chọn đáng tin cậy dành cho sinh viên và nhân viên văn phòng nhờ sự cân bằng hoàn hảo giữa hiệu suất và tính di động. Hiệu năng ổn định Sở hữu bộ vi xử lý Intel Core i3 1315U mạnh mẽ kết hợp cùng card đồ họa tích hợp Intel UHD Graphics, laptop Asus Vivobook mang đến hiệu năng vượt trội cho các công việc soạn thảo văn bản, xử lý số liệu trên Excel, ghi chú nhanh chóng và duyệt web mượt mà. Không chỉ vậy, người dùng cũng có thể chỉnh sửa ảnh cơ bản trên Photoshop, Canva hay giải trí với một số tựa game phổ thông mà không lo giật lag. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/334792/asus-vivobook-15-x1504va-i3-nj1634w-thumb-638760891054481940-600x600.jpg
acer-nitro-v-15-anv15-41-r2up-r5-nhqpgsv004,Acer Nitro V 15 ANV15 41 R2UP R5 6600H (NH.QPGSV.004),"Lưu ý: Tất cả sản phẩm gaming dòng Nitro V và Predator khi mua hàng có hoá đơn từ ngày 01/01/2025 trở đi, thời gian bảo hành máy thay đổi từ 1 năm thành 2 năm. Bạn đang tìm kiếm một chiếc laptop không chỉ mạnh mẽ trong cấu hình mà còn đậm cá tính nhưng giá thành lại còn hợp lý. Laptop Acer Nitro V 15 ANV15 41 R2UP R5 6600H (NH.QPGSV.004) chính là lựa chọn hoàn hảo để nâng tầm trải nghiệm game và sáng tạo của bạn. Sở hữu thiết kế ấn tượng cùng hiệu năng vượt trội, chiếc laptop này hứa hẹn sẽ chinh phục cả những game thủ khó tính nhất. Sức mạnh ""khủng"" bên trong Acer Nitro V 15 – Chiến mọi tác vụ thiết kế, kỹ thuật Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/333430/acer-nitro-v-15-anv15-41-r2up-r5-nhqpgsv004-638774737367845195-600x600.jpg
lenovo-ideapad-slim-3-15amn8-r5-82xq00j0vn,Lenovo Ideapad Slim 3 15AMN8 R5 7520U (82XQ00J0VN),"Trong thị trường laptop học tập - văn phòng, chiếc laptop Lenovo Ideapad Slim 3 15AMN8 R5 7520U (82XQ00J0VN) nhanh chóng thu hút sự chú ý của người dùng bởi nhiều ưu điểm nổi bật. Chiếc laptop này hội tụ đầy đủ các yếu tố để trở thành người bạn đồng hành lý tưởng cho học sinh, sinh viên và nhân viên văn phòng, đáp ứng mọi nhu cầu học tập, làm việc và giải trí một cách hiệu quả. • So với các đối thủ cùng phân khúc, chiếc laptop Lenovo Ideapad này sở hữu cấu hình khá ấn tượng và mới mẻ, đáp ứng tốt mọi nhu cầu học tập, làm việc và giải trí cơ bản. Với bộ vi xử lý AMD Ryzen 5 7520U có 4 nhân, 8 luồng và tốc độ xung nhịp tối đa lên đến 4.3 GHz. Con chip này mang đến hiệu năng mạnh mẽ, xử lý mượt mà các tác vụ học tập, văn phòng phổ biến như Word, Excel, PowerPoint,... Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/325500/lenovo-ideapad-slim-3-15amn8-r5-82xq00j0vn-thumb-638754862828598408-600x600.jpg
msi-thin-15-b12ucx-i5-2046vn,MSI Gaming Thin 15 B12UCX i5 12450H (2046VN),"Sự kết hợp hoàn hảo giữa bộ vi xử lý Intel Gen 12 mạnh mẽ và card đồ họa NVIDIA 20 series tiên tiến biến laptop MSI Gaming Thin 15 B12UCX i5 12450H (2046VN) thành cỗ máy chiến game và sáng tạo nội dung đích thực, đáp ứng mọi nhu cầu sử dụng của người dùng. • Laptop MSI Gaming Thin được trang bị bộ vi xử lý Intel Core i5 12450H với 8 nhân, 12 luồng cung cấp tốc độ xử lý ấn tượng giúp bạn hoàn thành mọi tác vụ một cách nhanh chóng và hiệu quả, dù là những công việc đòi hỏi khả năng tính toán cao hay những tựa game AAA nặng đô. • Card đồ họa NVIDIA GeForce RTX 2050 với bộ nhớ 4 GB cho bạn tận hưởng hình ảnh, đồ hoạ nét, mượt mà trong game và các ứng dụng đồ họa, đồng thời chiến mượt mọi tựa game yêu thích. • Dung lượng RAM 16 GB cho phép bạn mở nhiều ứng dụng cùng lúc, chuyển đổi qua lại linh hoạt mà không lo hiện tượng giật lag hay đơ máy. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/326124/msi-thin-15-b12ucx-i5-2046vn-140225-102530-055-600x600.jpg
hp-15-fd0303tu-i3-a2nl4pa,HP 15 fd0303TU i3 1315U (A2NL4PA),"Nếu bạn đang tìm kiếm một chiếc laptop HP cơ bản để phục vụ cho việc học hoặc làm việc văn phòng đến thiết kế đồ họa nhẹ nhàng thì còn chần chừ gì nữa mà không tham khảo ngay laptop HP 15 fd0303TU i3 1315U (A2NL4PA). Sản phẩm này ngoại hình hiện đại, gọn gàng sẵn sàng đồng hành cùng bạn ở bất kì đâu. • Với 6 nhân 8 luồng xử lý, chip Intel Core i3 Raptor Lake - 1315U dư sức xử lý mượt mà các tác vụ văn phòng hàng ngày như soạn thảo văn bản trên Word, bảng tính trên Excel, lập trình cơ bản,... Đi kèm với là card đồ họa tích hợp Intel UHD Graphics, cho phép bạn chỉnh sửa ảnh 2D cơ bản trên Photoshop, Canva,... một cách hiệu quả. • Laptop HP được trang bị RAM 8 GB giúp thực hiện đa nhiệm trơn tru, thoải mái mở nhiều ứng dụng cùng lúc mà không lo máy bị giật lag hay đơ. Bạn có thể mở nhiều tab Chrome, sử dụng các phần mềm văn phòng, chỉnh sửa ảnh,... đồng thời mà vẫn đảm bảo hiệu năng mượt mà. Xem thêm",laptop,https://cdnv2.tgdd.vn/mwg-static/tgdd/Products/Images/44/326050/hp-15-fd0303tu-i3-a2nl4pa-170225-110423-551-600x600.jpg"""

# Tạo DataFrame
products_df = pd.read_csv(pd.compat.StringIO(data))

# Hàm tiền xử lý văn bản (giả sử không có stopwords file)
def preprocess_text(data):
    data = data.lower()
    data = re.sub(r'\W+', ' ', data)
    data = word_tokenize(data, format="text")
    important_keywords = ['laptop', 'tai nghe', 'đồng hồ', 'điện thoại', 'asus', 'hp', 'dell', 'lenovo', 'acer', 'macbook', 'intel', 'amd', 'ryzen', 'core']
    data = ' '.join([word for word in data.split() if word in important_keywords])
    return data

# Tiền xử lý description
products_df['processed_description'] = products_df['description'].apply(preprocess_text)

# Tạo ma trận TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
description_matrix = vectorizer.fit_transform(products_df['processed_description'])

# Giả định người dùng với các sản phẩm đã đánh giá
user_rated_products = ['asus-vivobook-go-15-e1504fa-r5-nj776w', 'hp-15-fd0234tu-i5-9q969pa']
user_indices = [products_df.index[products_df['id_product'] == pid].tolist()[0] for pid in user_rated_products if pid in products_df['id_product'].values]

# Tính user profile
user_profile = np.asarray(description_matrix[user_indices].mean(axis=0))

# Mô phỏng Serendipity cho các giá trị near neighbors
neighbors = np.arange(5, 51, 5)  # Từ 5 đến 50
serendipity_scores = []

for n in neighbors:
    # Tính cosine similarity
    cosine_sim = cosine_similarity(user_profile, description_matrix)[0]
    # Lấy top n sản phẩm gần nhất (loại bỏ sản phẩm đã đánh giá)
    top_n_indices = cosine_sim.argsort()[::-1][len(user_indices):n+len(user_indices)]
    recommended_products = products_df.index[top_n_indices]
    
    # Mô phỏng Serendipity (dựa trên unexpectedness và relevance)
    similarities = cosine_sim[top_n_indices]
    unexpectedness = 1 - similarities
    relevance = np.where((similarities > 0.2) & (similarities < 0.8), 1.0, 0.0)
    serendipity = np.mean(unexpectedness * relevance)
    serendipity_scores.append(serendipity)

# Vẽ biểu đồ
plt.figure(figsize=(10, 6))
plt.plot(neighbors, serendipity_scores, marker='o', label='Content-based with location', color='blue')
plt.xlabel('Number of Near Neighbors')
plt.ylabel('Serendipity (0.0 - 1.0)')
plt.title('Serendipity vs Number of Near Neighbors')
plt.grid(True)
plt.legend()
plt.ylim(0, 1.0)
plt.show()